In [1]:
import sys
sys.path.append('../') 
from visualisation import *
import xarray as xr
import dask, re
import geopandas as gpd
from shapely.geometry import Point
from geopy.geocoders import Nominatim
from tqdm import tqdm  # For progress bar in large datasets
from geopy.exc import GeocoderTimedOut

In [2]:
bom_path = "/home/hossein/CICCADA/BOM_NCI/2023/01/01/"
files = glob(bom_path+"*.nc")
len(files)

103

In [4]:
df = [xr.open_dataset(file).to_dataframe() for file in files[:15]]
df = pd.concat(df, axis=0).reset_index(drop=False)
df = df.dropna(subset='direct_normal_irradiance').reset_index(drop=True)
df['julian_date'] = pd.to_datetime(df['julian_date'], origin='julian', unit='D')
df = df.query(f"latitude >= -35 & latitude <= -34.6 & longitude >= 138.5 & longitude <= 138.8").reset_index(drop=True)
df['geometry'] = [Point(x,y) for x,y in zip(df['longitude'], df['latitude'])]
geo_list = df['geometry'].unique()
print('len(geo_list): ', len(geo_list))

len(geo_list):  2259710


In [5]:
i = 0

In [6]:
address_i = []
i_i = []
for _ in range(len(geo_list)):
    try:
        address = gpd.tools.reverse_geocode(geo_list[i], timeout=60)
        if address['address'].to_list()[0]!=None:
            print(address['address'], i)
            address_i.append(address)
            i_i.append(i)
    except Exception as e:
        pass
        print(e, i)
    i += 1


0    271, Brighton Road, 5044, Brighton Road, City ...
Name: address, dtype: object 1
0    Kent Avenue, 5046, City of Marion, South Austr...
Name: address, dtype: object 2
0    Sampson Road, 5043, City of Marion, South Aust...
Name: address, dtype: object 3
0    Styles Avenue, 5042, City of Mitcham, South Au...
Name: address, dtype: object 4
0    Crescent Avenue, 5041, City of Mitcham, South ...
Name: address, dtype: object 5
0    Gloucester Avenue, 5052, City of Mitcham, Sout...
Name: address, dtype: object 6
0    Brady Gully Track, 5052, City of Mitcham, Sout...
Name: address, dtype: object 7
0    Wilyawa Track, 5052, City of Mitcham, South Au...
Name: address, dtype: object 8
0    Charlick Road, 5156, City of Mitcham, South Au...
Name: address, dtype: object 9
0    5, Atkinson Road, 5152, Atkinson Road, Adelaid...
Name: address, dtype: object 10
0    Unnamed (No.HA1420) Heritage Agreement, 5152, ...
Name: address, dtype: object 11
0    St Margaret Drive, 5152, Stirling, South Austr.

In [14]:
postcode = re.search(r'\b\d{4}\b', address_i[0]['address'].to_list()[0])
postcode.group()

'5044'

In [53]:
i_i[0], address_i[0]['address'].to_list()[0]

(1,
 '271, Brighton Road, 5044, Brighton Road, City of Holdfast Bay, South Australia, Australia')

In [67]:
len(address_i), len(i_i)

(324, 324)

In [68]:
postcode_i = [re.search(r'\b\d{4}\b', i['address'].to_list()[0]) for i in address_i]
postcode_i = [i.group() if i is not None else 'None' for i in postcode_i]
lat_i = [geo_list[i].y for i in i_i]
lon_i = [geo_list[i].x for i in i_i]



In [70]:
df_address = pd.DataFrame({'postcode': postcode_i, 'latitude': lat_i, 'longitude': lon_i})

In [76]:
df.merge(df_address, on=['latitude', 'longitude'], how='inner')

,time,latitude,longitude,crs,surface_global_irradiance,direct_normal_irradiance,surface_diffuse_irradiance,quality_mask,cloud_type,cloud_optical_depth,solar_elevation,solar_azimuth,julian_date,geometry,postcode
0,2022-12-31 19:30:00,-35.000000,138.520004,-2147483647,0.00,0.00,0.00,1.0,0.0,0.0,-0.3,118.8,2022-12-31 19:37:58.876139225,POINT (138.52000427246094 -35),5044.0
1,2022-12-31 19:30:00,-35.000000,138.539993,-2147483647,0.00,0.00,0.00,1.0,0.0,0.0,-0.3,118.8,2022-12-31 19:37:58.876139225,POINT (138.5399932861328 -35),5046.0
2,2022-12-31 19:30:00,-35.000000,138.559998,-2147483647,0.00,0.00,0.00,1.0,0.0,0.0,-0.3,118.8,2022-12-31 19:37:58.876139225,POINT (138.55999755859375 -35),5043.0
3,2022-12-31 19:30:00,-35.000000,138.580002,-2147483647,0.00,0.00,0.00,1.0,0.0,0.0,-0.3,118.8,2022-12-31 19:37:58.876139225,POINT (138.5800018310547 -35),5042.0
4,2022-12-31 19:30:00,-35.000000,138.600006,-2147483647,0.00,0.00,0.00,1.0,0.0,0.0,-0.2,118.8,2022-12-31 19:37:58.876139225,POINT (138.60000610351562 -35),5041.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2911,2022-12-31 21:20:00,-34.599998,138.720001,-2147483647,329.79,773.72,56.93,1.0,0.0,0.0,20.7,104.4,2022-12-31 21:27:57.246786650,POINT (138.72000122070312 -34.599998474121094),5118.0
2912,2022-12-31 21:20:00,-34.599998,138.740005,-2147483647,330.33,774.65,56.91,1.0,0.0,0.0,20.7,104.4,2022-12-31 21:27:57.246786650,POINT (138.74000549316406 -34.599998474121094),5118.0
2913,2022-12-31 21:20:00,-34.599998,138.759995,-2147483647,330.79,779.60,55.40,1.0,0.0,0.0,20.7,104.4,2022-12-31 21:27:57.246786650,POINT (138.75999450683594 -34.599998474121094),5118.0
2914,2022-12-31 21:20:00,-34.599998,138.779999,-2147483647,332.38,784.38,55.07,1.0,0.0,0.0,20.7,104.3,2022-12-31 21:27:57.246786650,POINT (138.77999877929688 -34.599998474121094),5118.0


In [72]:
df_address.to_csv('Adelide_postcode_points.csv', index=False)

In [73]:
df_address = pd.read_csv('Adelide_postcode_points.csv')

In [74]:
df_address

,postcode,latitude,longitude
0,5044.0,-35.000000,138.520004
1,5046.0,-35.000000,138.539993
2,5043.0,-35.000000,138.559998
3,5042.0,-35.000000,138.580002
4,5041.0,-35.000000,138.600006
...,...,...,...
319,5118.0,-34.599998,138.720001
320,5118.0,-34.599998,138.740005
321,5118.0,-34.599998,138.759995
322,5118.0,-34.599998,138.779999


In [61]:
df[df['latitude']==geo_list[1].y]

,time,latitude,longitude,crs,surface_global_irradiance,direct_normal_irradiance,surface_diffuse_irradiance,quality_mask,cloud_type,cloud_optical_depth,solar_elevation,solar_azimuth,julian_date,geometry
0,2022-12-31 19:30:00,-35.0,138.500000,-2147483647,0.00,0.00,0.00,4.0,0.0,0.0,-0.3,118.8,2022-12-31 19:37:58.876139225,POINT (138.5 -35)
1,2022-12-31 19:30:00,-35.0,138.520004,-2147483647,0.00,0.00,0.00,1.0,0.0,0.0,-0.3,118.8,2022-12-31 19:37:58.876139225,POINT (138.52000427246094 -35)
2,2022-12-31 19:30:00,-35.0,138.539993,-2147483647,0.00,0.00,0.00,1.0,0.0,0.0,-0.3,118.8,2022-12-31 19:37:58.876139225,POINT (138.5399932861328 -35)
3,2022-12-31 19:30:00,-35.0,138.559998,-2147483647,0.00,0.00,0.00,1.0,0.0,0.0,-0.3,118.8,2022-12-31 19:37:58.876139225,POINT (138.55999755859375 -35)
4,2022-12-31 19:30:00,-35.0,138.580002,-2147483647,0.00,0.00,0.00,1.0,0.0,0.0,-0.3,118.8,2022-12-31 19:37:58.876139225,POINT (138.5800018310547 -35)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2699,2022-12-31 21:20:00,-35.0,138.720001,-2147483647,348.10,836.80,51.46,1.0,0.0,0.0,20.8,104.2,2022-12-31 21:27:58.876148160,POINT (138.72000122070312 -35)
2700,2022-12-31 21:20:00,-35.0,138.740005,-2147483647,346.43,829.07,52.29,1.0,0.0,0.0,20.8,104.2,2022-12-31 21:27:58.876148160,POINT (138.74000549316406 -35)
2701,2022-12-31 21:20:00,-35.0,138.759995,-2147483647,345.46,825.71,52.27,1.0,0.0,0.0,20.8,104.2,2022-12-31 21:27:58.876148160,POINT (138.75999450683594 -35)
2702,2022-12-31 21:20:00,-35.0,138.779999,-2147483647,343.31,816.27,53.23,1.0,0.0,0.0,20.8,104.2,2022-12-31 21:27:58.876148160,POINT (138.77999877929688 -35)


In [ ]:

# Add a reverse geocoding function
def get_postcode(point):
    location = geolocator.reverse((point.y, point.x), exactly_one=True)
    if location and 'postcode' in location.raw['address']:
        return location.raw['address']['postcode']
    return None

# Apply with progress bar
tqdm.pandas()
gdf['Postcode'] = gdf['geometry'].progress_apply(get_postcode)

print(gdf)

In [1]:
np.sort(df0.loc[0, 'julian_date'])

NameError: name 'np' is not defined

In [27]:
np.sort(df0.loc[10, 'julian_date'])

array(['2020-11-19T18:38:35.534292158', '2020-11-19T18:48:35.534296630',
       '2020-11-19T18:58:35.534301101', '2020-11-19T19:08:35.534305573',
       '2020-11-19T19:18:35.534310045', '2020-11-19T19:28:35.534274280',
       '2020-11-19T19:38:35.534278752', '2020-11-19T19:48:35.534283223',
       '2020-11-19T19:58:35.534287695'], dtype='datetime64[ns]')

In [28]:
np.sort(df0.loc[22, 'time'])

array(['2020-11-19T18:30:00.000000000', '2020-11-19T18:40:00.000000000',
       '2020-11-19T18:50:00.000000000', '2020-11-19T19:00:00.000000000',
       '2020-11-19T19:10:00.000000000', '2020-11-19T19:20:00.000000000',
       '2020-11-19T19:30:00.000000000', '2020-11-19T19:40:00.000000000',
       '2020-11-19T19:50:00.000000000'], dtype='datetime64[ns]')

In [ ]:

# print(df.columns)
# print(df['julian_date'][0])
df = df.query('direct_normal_irradiance > 0').drop_duplicates(subset='julian_date').loc[:, ['julian_date', 'direct_normal_irradiance', 'surface_global_irradiance',
 'surface_diffuse_irradiance', 'cloud_optical_depth', 'cloud_type']]
print(df['julian_date'].min(), df['julian_date'].max())
print(df[:30])